# MMUbiPred robust-loss sensitivity v1

This development-only notebook compares BCE, asymmetric negative-label smoothing (`0.05`), and generalized cross entropy (`q=0.7`) using the identical MMUbiPred-compatible architecture and seed-42 protein-grouped split. It never loads the released independent or external test sets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os, subprocess, sys
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

GITHUB_TOKEN = secret('GITHUB_TOKEN')
git_env = os.environ.copy()
git_env['GIT_TERMINAL_PROMPT'] = '0'
askpass = Path('/tmp/ubipred_git_askpass.sh')
if GITHUB_TOKEN:
    askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token;; *) echo "$GITHUB_TOKEN";; esac\n')
    askpass.chmod(0o700)
    git_env.update({'GIT_ASKPASS': str(askpass), 'GITHUB_TOKEN': GITHUB_TOKEN})

REPO_ROOT = Path('/content/Ubiquitination-Sites-Prediction')
try:
    if not (REPO_ROOT / '.git').exists():
        subprocess.run(['git', 'clone', '--recurse-submodules', 'https://github.com/kaleabnega/Ubiquitination-Sites-Prediction.git', str(REPO_ROOT)], check=True, env=git_env)
    else:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True, env=git_env)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'submodule', 'update', '--init', '--recursive'], check=True, env=git_env)
finally:
    askpass.unlink(missing_ok=True)
os.chdir(REPO_ROOT)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

In [ ]:
%pip install -q -r experiment/requirements-colab.txt
%pip uninstall -y torchao

In [ ]:
import json, torch

if not torch.cuda.is_available():
    raise RuntimeError('A GPU runtime is required for training.')
print('GPU:', torch.cuda.get_device_name(0))

def run_live(command):
    command = [str(part) for part in command]
    print('$', ' '.join(command), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

In [ ]:
SUITE = Path('experiment/configs/mmubipred_robust_loss_sensitivity_v1.json')
OUTPUT_ROOT = Path('/content/drive/MyDrive/MMUbiPred-research/benchmarks/mmubipred-robust-loss-sensitivity-v1')
SUMMARY_PATH = OUTPUT_ROOT / 'summary.json'
DECISION_PATH = OUTPUT_ROOT / 'screen_decision.json'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Persistent output:', OUTPUT_ROOT)

## Run or resume the three paired conditions

Each condition runs 15 epochs. Completed conditions are skipped, and an interrupted condition resumes from its last completed epoch.

In [ ]:
run_live([
    sys.executable, '-u', 'experiment/scripts/run_validation_benchmark.py',
    '--suite', SUITE,
    '--output-root', OUTPUT_ROOT,
    '--resume',
])

In [ ]:
if not DECISION_PATH.exists():
    run_live([
        sys.executable, '-u', 'experiment/scripts/analyze_robust_loss_screen.py',
        '--suite', SUITE,
        '--summary', SUMMARY_PATH,
        '--output', DECISION_PATH,
    ])
else:
    print('Existing completed screen decision found; it was not repeated.')

summary = json.loads(SUMMARY_PATH.read_text())
decision = json.loads(DECISION_PATH.read_text())
display(summary['runs'])
display({
    'comparison_valid': decision['comparison_valid'],
    'candidate_results': decision['candidate_results'],
    'decision': decision['decision'],
    'selected_candidate': decision['selected_candidate'],
    'next_step': decision['next_step'],
    'released_independent_test_accessed': decision['released_independent_test_accessed'],
    'external_test_accessed': decision['external_test_accessed'],
})

## Development decision record

The two displayed outputs form the decision record required before additional seeds or loss-parameter changes are considered.